# 01 — Data Exploration
EDA of the ISIC melanoma dataset: class balance, sample images, and pixel statistics.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

ROOT     = Path('..')
DATA_DIR = ROOT / 'data'
df = pd.read_csv(DATA_DIR / 'metadata.csv')
df['class_name'] = df['label'].map({0: 'Benign', 1: 'Malignant'})
print(f"Total samples : {len(df)}")
print(df['class_name'].value_counts())
df.head()

## Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['class_name'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Class Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Class Balance', fontsize=13, fontweight='bold')

plt.suptitle('ISIC Melanoma Dataset — Class Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(f"Class balance ratio: {counts.min()/counts.max():.2f} (1.0 = perfect balance)")

## Sample Images per Class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for cls_idx, (label, cls_name) in enumerate([(0,'Benign'), (1,'Malignant')]):
    samples = df[df['label'] == label].sample(5, random_state=42)
    for col, (_, row) in enumerate(samples.iterrows()):
        img_path = DATA_DIR / (row['image_id'] + '.jpg')
        img = Image.open(img_path).resize((224, 224))
        axes[cls_idx][col].imshow(img)
        axes[cls_idx][col].set_title(cls_name, color='green' if label==0 else 'red',
                                     fontsize=9, fontweight='bold')
        axes[cls_idx][col].axis('off')

plt.suptitle('Sample Dermoscopy Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Pixel Intensity Statistics per Class

In [ ]:
stats = []
for _, row in df.sample(100, random_state=42).iterrows():
    img_path = DATA_DIR / (row['image_id'] + '.jpg')
    arr = np.array(Image.open(img_path).resize((64, 64)))
    stats.append({'label': row['class_name'],
                  'mean_r': arr[:,:,0].mean(), 'mean_g': arr[:,:,1].mean(),
                  'mean_b': arr[:,:,2].mean(), 'std': arr.std()})

stats_df = pd.DataFrame(stats)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col, color in zip(axes, ['mean_r','mean_g','mean_b','std'],
                          ['#e74c3c','#2ecc71','#3498db','#9b59b6']):
    stats_df.groupby('label')[col].plot(kind='density', ax=ax, color=color, legend=True)
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.set_xlabel('Pixel Value')

plt.suptitle('Pixel Intensity Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(stats_df.groupby('label').agg({'mean_r':'mean','mean_g':'mean','mean_b':'mean','std':'mean'}).round(2))